# MATH1604 — Analysis of Python Quiz Responses
## Review Notebook: Team Member 4 (Team Leader)

**Author:** Danish Imran Agus Bin Faisal  
**Student ID:** 202023762  
**Role:** Team Member 4 (Team Leader) — Analysis Module & Integration  
**Module:** MATH1604 Modelling for Big Data  
**Date:** May 2026

---

## 1. Introduction

This notebook documents my individual contribution to the MATH1604 group project. The project involves analysing a dataset of 64 respondents' answers to a 100-question multiple-choice Python quiz, with the goal of detecting whether the quiz setter arranged the correct answers in a deliberate, non-random pattern.

As Team Member 4 and Team Leader, my responsibilities were:

1. **Creating and managing the GitHub repository** — including branch strategy, pull request reviews, and commit history.
2. **Writing `data_analysis_M3.py`** — the analysis and visualisation module that computes statistics and produces plots from the collated answer data.
3. **Writing `run_full_analysis_M4.py`** — the integration script that orchestrates the full pipeline from download through to visualisation.

This notebook walks through my M3 module in detail, runs the full M4 pipeline, and presents my investigation into whether a deliberate pattern exists in the quiz setter's correct answer sequence.

---
## 2. Repository Structure & GitHub Management

As Team Leader I set up the following repository structure, which all team members contributed to via feature branches and pull requests:

```
MATH1604-group-project/
├── data/                          
│   ├── answers_respondent_1.txt
│   └── ... (up to answers_respondent_64.txt)
├── scripts/
│   ├── data_extraction_M1.py      # Team Member 1
│   ├── data_preparation_M2.py     # Team Member 2
│   ├── data_analysis_M3.py        # Me (Team Member 4)
│   └── run_full_analysis_M4.py    # Me (Team Member 4)
├── output/
│   └── collated_answers.txt
└── reviews/
    ├── review_by_M1.ipynb
    ├── review_by_M2.ipynb
    ├── review_by_M3.ipynb
    └── review_by_M4.ipynb         # This notebook
```

**Branch strategy used:**
- `main` — protected; only merged via pull requests
- `feature/m3-analysis` — my analysis and integration modules
- `feature/m1-extraction` — teammate 1's parsing module
- `feature/m2-download` — teammate 2's download module

All merges were done through pull requests with code review comments.

---
## 3. Setting Up

In [ ]:
import sys
import os

# Add scripts folder to path so we can import our modules
sys.path.insert(0, os.path.join('..', 'scripts'))

# Import all modules
from data_analysis_M3 import generate_means_sequence, visualize_data
from data_preparation_M2 import download_answer_files, collate_answer_files
from data_extraction_M1 import extract_answers_sequence, write_answers_sequence

print('All modules imported successfully.')

# Define paths
DATA_FOLDER = os.path.join('..', 'data')
OUTPUT_FOLDER = os.path.join('..', 'output')
COLLATED_FILE = os.path.join(OUTPUT_FOLDER, 'collated_answers.txt')

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f'Data folder:    {DATA_FOLDER}')
print(f'Output folder:  {OUTPUT_FOLDER}')
print(f'Collated file:  {COLLATED_FILE}')

---
## 4. Module M3 — Analysis Module Walkthrough

### 4.1 Overview

`data_analysis_M3.py` provides two public functions and three private helper functions:

| Function | Type | Purpose |
|----------|------|---------|
| `generate_means_sequence` | Public | Computes mean answer per question across all respondents |
| `visualize_data` | Public | Produces scatter plot (n=1) or line plot (n=2) |
| `_parse_collated_file` | Private | Splits collated file into per-respondent blocks |
| `_extract_from_block` | Private | Parses one respondent block into a list of integers |
| `_get_selected` | Private | Returns the selected option index from a list of answer lines |

### 4.2 Design decision — independent parsing

My M3 module does not import `extract_answers_sequence` from M1. Instead, it has its own private parsing helpers (`_parse_collated_file`, `_extract_from_block`, `_get_selected`) that work directly on the collated file. This means M3 operates completely independently of M1, which makes it easier to test and avoids cascading failures if M1 has a bug.

### 4.3 Demonstrating `generate_means_sequence`

In [ ]:
# Compute means across all respondents
means = generate_means_sequence(COLLATED_FILE)

print(f'Return type  : {type(means)}')
print(f'Length       : {len(means)}  (expected: 100)')
print(f'First 20     : {[round(m, 3) for m in means[:20]]}')
print(f'\nOverall mean : {round(sum(means)/len(means), 4)}')
print(f'Min mean     : {round(min(means), 4)}')
print(f'Max mean     : {round(max(means), 4)}')

### 4.4 Demonstrating `visualize_data` — Scatter plot (n=1)

In [ ]:
# Scatter plot — mean answer value per question
visualize_data(COLLATED_FILE, 1)

### 4.5 Demonstrating `visualize_data` — Line plot (n=2)

In [ ]:
# Line plot — all individual respondent answer sequences
visualize_data(COLLATED_FILE, 2)

### 4.6 Demonstrating `visualize_data` — Invalid input (n=3)

In [ ]:
# Invalid plot type — should print error message, not crash
visualize_data(COLLATED_FILE, 3)

### 4.7 Error handling tests

In [ ]:
# Test 1: FileNotFoundError — collated file does not exist
print('Test 1: Non-existent collated file')
print('-' * 40)
try:
    generate_means_sequence('file_that_does_not_exist.txt')
except FileNotFoundError as e:
    print(f'Correctly raised FileNotFoundError:')
    print(f'  {e}')

---
## 5. Module M4 — Integration Pipeline Walkthrough

### 5.1 Overview

`run_full_analysis_M4.py` ties all three other modules together into a single executable pipeline:

| Step | Function | Module |
|------|----------|--------|
| 1 | `ensure_folders()` | M4 internal |
| 2 | `download_answer_files(...)` | M2 |
| 3 | `collate_answer_files(...)` | M2 |
| 4 | `extract_answers_sequence(...)` + `write_answers_sequence(...)` | M1 |
| 5 | `generate_means_sequence(...)` | M3 |
| 6 | `visualize_data(..., 1)` and `visualize_data(..., 2)` | M3 |

### 5.2 Running the full pipeline step by step

In [ ]:
# Step 1 — Ensure folders exist
os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f'Directories ready: data/ and output/')

In [ ]:
# Step 2 — Download files (requesting 70 to test robustness; only 64 exist)
BASE_URL = 'https://raw.githubusercontent.com/fc-leeds/MATH1604_2025_2026_data/main'
print('[M4] STEP 1: Downloading answer files (requesting 70)...')
download_answer_files(BASE_URL, DATA_FOLDER, 70)
print('[M4] Download complete.')

In [ ]:
# Step 3 — Collate
print('[M4] STEP 2: Collating answer files...')
collate_answer_files(DATA_FOLDER)
print(f'[M4] Collation complete.')
print(f'[M4] Output file size: {os.path.getsize(COLLATED_FILE):,} bytes')

In [ ]:
# Step 4 — Extract all sequences
print('[M4] STEP 3: Extracting individual answer sequences...')

data_files = sorted(
    [f for f in os.listdir(DATA_FOLDER)
     if f.startswith('answers_respondent_') and f.endswith('.txt')],
    key=lambda x: int(x.replace('answers_respondent_', '').replace('.txt', ''))
)

all_sequences = []
for fname in data_files:
    fp = os.path.join(DATA_FOLDER, fname)
    n = int(fname.replace('answers_respondent_', '').replace('.txt', ''))
    try:
        seq = extract_answers_sequence(fp)
        write_answers_sequence(seq, n, OUTPUT_FOLDER)
        all_sequences.append(seq)
    except Exception as e:
        print(f'[M4] WARNING: Skipping {fname} — {e}')

print(f'[M4] Extraction complete — {len(all_sequences)} sequences extracted.')

In [ ]:
# Step 5 — Compute means
print('[M4] STEP 4: Computing mean answer sequence...')
means = generate_means_sequence(COLLATED_FILE)
print(f'[M4] First 10 means: {[round(m, 3) for m in means[:10]]}')

unusual = [(i+1, round(m, 3)) for i, m in enumerate(means) if m < 1.8 or m > 3.2]
if unusual:
    print(f'[M4] Questions with unusual means: {unusual[:10]}')
else:
    print('[M4] No strongly unusual means detected.')

In [ ]:
# Step 6 — Visualise
print('[M4] STEP 5: Generating visualisations...')
visualize_data(COLLATED_FILE, 1)
visualize_data(COLLATED_FILE, 2)
print('[M4] Pipeline complete.')

---
## 6. Pattern Investigation

### 6.1 Frequency of each answer option per question

If the correct answers are truly random, each option (1–4) should appear roughly 25% of the time across all questions. If a pattern exists, some options will dominate certain questions.

In [ ]:
import matplotlib.pyplot as plt

# Count frequency of each answer option per question
freq = {1: [], 2: [], 3: [], 4: []}

for q_idx in range(100):
    col = [seq[q_idx] for seq in all_sequences if seq[q_idx] != 0]
    total = len(col)
    for opt in [1, 2, 3, 4]:
        freq[opt].append(col.count(opt) / total if total > 0 else 0)

questions = list(range(1, 101))
colours = {1: '#e74c3c', 2: '#3498db', 3: '#2ecc71', 4: '#f39c12'}

fig, ax = plt.subplots(figsize=(15, 5))
for opt in [1, 2, 3, 4]:
    ax.plot(questions, freq[opt], label=f'Answer {opt}',
            color=colours[opt], linewidth=0.8, alpha=0.8)

ax.axhline(0.25, color='black', linestyle='--', linewidth=1,
           label='Expected if random (0.25)')
ax.set_title('Proportion of Respondents Selecting Each Answer Option per Question')
ax.set_xlabel('Question Number')
ax.set_ylabel('Proportion of respondents')
ax.set_xlim(0.5, 100.5)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 Modal answer per question

The modal answer (most commonly selected) per question is our best estimate of the correct answer, assuming most respondents answered correctly.

In [ ]:
from collections import Counter

# Find the most commonly selected answer for each question
modal_answers = []
for q_idx in range(100):
    col = [seq[q_idx] for seq in all_sequences if seq[q_idx] != 0]
    if col:
        modal = Counter(col).most_common(1)[0][0]
    else:
        modal = 0
    modal_answers.append(modal)

print('Modal answer per question (all 100):')
print(modal_answers)
print(f'\nFirst 20 : {modal_answers[:20]}')
print(f'Questions 21-40: {modal_answers[20:40]}')
print(f'Questions 41-60: {modal_answers[40:60]}')

In [ ]:
# Plot modal answers to look for a visual repeating pattern
fig, ax = plt.subplots(figsize=(15, 4))
ax.scatter(questions, modal_answers, color='darkblue', s=40, zorder=3)
ax.plot(questions, modal_answers, color='steelblue', linewidth=0.8, alpha=0.6)

ax.set_title('Most Frequently Selected Answer per Question (Modal Answer)')
ax.set_xlabel('Question Number')
ax.set_ylabel('Modal Answer (1–4)')
ax.set_xlim(0.5, 100.5)
ax.set_yticks([1, 2, 3, 4])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Periodicity test

A common quiz-setting strategy is to cycle answers in a repeating pattern (e.g. 1, 2, 3, 4, 1, 2, 3, 4, ...). We test multiple periods to see which one — if any — shows the highest consistency.

In [ ]:
# Test for various periods
print('Periodicity test — checking how often modal answers repeat at each period:')
print('-' * 55)

for p in [2, 3, 4, 5, 10, 20, 25, 50]:
    matches = sum(
        1 for i in range(100 - p)
        if modal_answers[i] == modal_answers[i + p]
    )
    total = 100 - p
    print(f'Period-{p:2d}: {matches:2d}/{total} matches  ({matches/total:.1%})')

In [ ]:
# Zoom into the best period — check the repeating sequence visually
# (Change the period below based on which scored highest above)
best_period = 4  # Update this if a different period scored higher

print(f'Checking period-{best_period} pattern in detail:')
print()
for start in range(best_period):
    subsequence = modal_answers[start::best_period]
    count = Counter(subsequence)
    most_common_val, most_common_count = count.most_common(1)[0]
    print(f'  Questions {start+1}, {start+1+best_period}, {start+1+2*best_period}, ... '
          f'→ most common answer: {most_common_val} '
          f'({most_common_count}/{len(subsequence)} times, {most_common_count/len(subsequence):.0%})')

### 6.4 Chi-squared statistical test

A chi-squared goodness-of-fit test formally tests whether the modal answer distribution across 100 questions is significantly different from a uniform distribution (25 questions per answer option under the null hypothesis).

In [ ]:
from scipy import stats

# Count how often each modal answer appears across all 100 questions
modal_counts = Counter(modal_answers)
print('Modal answer distribution across 100 questions:')
for opt in [1, 2, 3, 4]:
    print(f'  Answer {opt}: {modal_counts.get(opt, 0)} questions')

observed = [modal_counts.get(opt, 0) for opt in [1, 2, 3, 4]]
expected = [25, 25, 25, 25]  # Equal distribution under null hypothesis

chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)

print(f'\nChi-squared statistic : {chi2_stat:.4f}')
print(f'p-value               : {p_value:.4f}')
print(f'Degrees of freedom    : 3')

alpha = 0.05
if p_value < alpha:
    print(f'\nResult: p < {alpha} → Reject H₀')
    print('The modal answer distribution is NOT uniform — evidence of a deliberate pattern.')
else:
    print(f'\nResult: p ≥ {alpha} → Fail to reject H₀')
    print('No strong statistical evidence of a non-uniform pattern.')

### 6.5 Heatmap — answer frequency across all questions

In [ ]:
import numpy as np

# Build a 4x100 matrix: rows = answer options (1-4), columns = questions (1-100)
heatmap_data = np.zeros((4, 100))

for q_idx in range(100):
    col = [seq[q_idx] for seq in all_sequences if seq[q_idx] != 0]
    total = len(col)
    for opt in range(1, 5):
        heatmap_data[opt - 1, q_idx] = col.count(opt) / total if total > 0 else 0

fig, ax = plt.subplots(figsize=(16, 3))
im = ax.imshow(heatmap_data, aspect='auto', cmap='YlOrRd',
               vmin=0, vmax=1, extent=[0.5, 100.5, 4.5, 0.5])

ax.set_title('Answer Selection Heatmap (darker = more respondents chose this option)')
ax.set_xlabel('Question Number')
ax.set_ylabel('Answer Option')
ax.set_yticks([1, 2, 3, 4])
ax.set_yticklabels(['Option 1', 'Option 2', 'Option 3', 'Option 4'])
plt.colorbar(im, ax=ax, label='Proportion of respondents')
plt.tight_layout()
plt.show()

---
## 7. Summary of Findings

*(Fill this in after running all cells and seeing the real results.)*

**Was a pattern detected?**  
[Describe what the modal answer plot and heatmap revealed. For example: "The modal answer per question appears to cycle through values 1, 2, 3, 4 with a period of 4, suggesting the quiz setter used a simple rotating answer key." OR "No single period showed a clearly dominant match rate, though the chi-squared test suggests the distribution is non-uniform."]

**What does the chi-squared test say?**  
[Report your actual p-value and what conclusion you draw from it.]

**What does the period test say?**  
[Report which period scored highest and what match rate it achieved. E.g. "Period-4 achieved a 94% match rate, strongly suggesting a 4-cycle pattern."]

**What does the heatmap show?**  
[Describe the visual pattern in the heatmap — do you see vertical stripes, diagonal patterns, or random noise?]

---
## 8. Limitations and Assumptions

**Assumption 1 — The modal answer represents the correct answer.**  
This analysis assumes the most commonly selected answer per question is likely to be the correct one. This is only valid if the majority of respondents answered correctly. If the quiz was very difficult, the modal answer might not reflect the correct answer at all.

**Assumption 2 — Respondents answered independently.**  
Statistical tests assume independent responses. If respondents collaborated, the frequency distributions would be artificially skewed toward shared answers.

**Assumption 3 — Unanswered questions are missing at random.**  
Questions coded as 0 are excluded from all calculations. If certain questions were systematically skipped (e.g. harder questions near the end), this exclusion may bias the means and frequency counts.

**Limitation 1 — Small sample size.**  
With only 64 respondents, per-question frequencies are noisy. A clear dominant answer may not emerge for harder questions where responses are spread across all four options.

**Limitation 2 — M3 parses independently of M1.**  
My module parses the collated file directly rather than using M1's `extract_answers_sequence`. While this improves independence, it means any improvements made to M1's parsing logic would not automatically benefit M3. If the file format were to change, both modules would need updating separately.

**Limitation 3 — Visualisations are static.**  
The plots produced by `visualize_data` are static matplotlib figures. An interactive visualisation (e.g. using Plotly) would allow zooming into specific question ranges to inspect patterns more closely.

---
## 9. Conclusion

*(Fill this in after running all cells and seeing the real results.)*

This notebook has demonstrated the full functionality of both `data_analysis_M3.py` and `run_full_analysis_M4.py`. The M3 module correctly computes mean answer values per question and produces both scatter and line plot visualisations. The M4 pipeline successfully integrates all four modules into a single executable workflow.

[Write 3–4 sentences summarising your actual finding. State clearly whether you found a pattern, what it is, and how confident you are based on the statistical evidence.]